In [6]:
import copy
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image

# Suppress warnings
import warnings

warnings.filterwarnings("ignore")

print("All imports loaded successfully!")

All imports loaded successfully!


In [ ]:
BASE_DIR = Path("..").resolve()

if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

PROCESSED_DIR = BASE_DIR / "data" / "cv" / "processed"
OUTPUTS_DIR = BASE_DIR / "outputs"
FIGURES_DIR = BASE_DIR / "figures"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running EfficientNet-B0 pipeline on device: {device}")

Running EfficientNet-B0 pipeline on device: cpu


In [ ]:
class CrashDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = BASE_DIR / row["frame_path"]
        image = Image.open(img_path).convert("RGB")
        target = float(row["label"])
        video_id = str(row["video_id"])

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(target, dtype=torch.float32), video_id


# Standard ImageNet transforms
train_transform = transforms.Compose(
    [
        transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

train_dataset = CrashDataset(
    PROCESSED_DIR / "", transform=train_transform
)
val_dataset = CrashDataset(PROCESSED_DIR / "", transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)

print(
    f"Loaded {len(train_dataset)} training frames and {len(val_dataset)} validation frames."
)

Loaded 1200 training frames and 300 validation frames.


In [9]:
# Import train_run from engine.py (assuming engine.py is updated with backbone support)
from src.cv.engine import build_model

# Test building EfficientNet-B0
test_model = build_model(dropout=0.2, device=device, backbone="efficientnet_b0")
print("EfficientNet-B0 successfully built and moved to device!")
print(test_model.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\98936/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:13<00:00, 1.53MB/s]


EfficientNet-B0 successfully built and moved to device!
Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=1, bias=True)
)


In [ ]:
from src.cv.engine import train_run


def objective(trial):
    params = {
        "backbone": "efficientnet_b0",  
        "lr": trial.suggest_float("lr", 1e-4, 5e-3, log=True),
        "dropout": trial.suggest_float("dropout", 0.1, 0.5),
        "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True),
        "batch_size": trial.suggest_categorical("batch_size", [16, 32]),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW"]),
    }
    epochs = trial.suggest_int("epochs", 4, 10)

    criterion = nn.BCEWithLogitsLoss()

    try:
        _, _, best_f1, _ = train_run(
            params=params,
            epochs=epochs,
            train_dataset=train_dataset,
            val_loader=val_loader,
            criterion=criterion,
            device=device,
            trial=trial,
            patience=3,
            label_smoothing=0.1,
        )
    except optuna.TrialPruned:
        raise
    except Exception as e:
        print(f"Trial failed with error: {e}")
        return 0.0

    return best_f1


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=2),
)

print("Starting Optuna HPO for EfficientNet-B0...")
study.optimize(objective, n_trials=8, show_progress_bar=True)

print("\nBest trial:")
print(f"  Value (Val Video Macro F1): {study.best_value:.4f}")
print("  Best params:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

In [ ]:
best_params = study.best_params.copy()
best_epochs = best_params.pop("epochs", 8)

# Ensure backbone is set to efficientnet_b0
best_params["backbone"] = "efficientnet_b0"

print(
    f"Training final EfficientNet-B0 model with best params for {best_epochs} epochs..."
)
criterion = nn.BCEWithLogitsLoss()

model_efficientnet, history, best_val_f1, best_state = train_run(
    params=best_params,
    epochs=best_epochs,
    train_dataset=train_dataset,
    val_loader=val_loader,
    criterion=criterion,
    device=device,
    trial=None,
    patience=3,
    label_smoothing=0.1,
)

# Save checkpoint with distinct name
model_path = OUTPUTS_DIR / "model_2_efficientnet_b0_tuned.pth"
torch.save(best_state, model_path)
print(f"Final EfficientNet-B0 model saved successfully at: {model_path}")
print(f"Best Validation Video Macro F1: {best_val_f1:.4f}")